In [15]:
import json
import os
import classify_paper
import pandas as pd
from typing import Tuple

PRINT_RANK="A*"

crossref_mapping = {}

def get_core_rank(pub: dict) -> Tuple[str, bool, object]:
    year = int(pub.get("year", 0))
    # Extract venue key from paper key or crossref
    venue_crossref = pub.get("crossref", "")
    venue_dblp = ""
    if venue_crossref:
        venue_dblp = '/'.join(venue_crossref.split('/')[:2])
    else:
        key = pub.get("key", "")
        if key:
            venue_dblp = '/'.join(key.split('/')[:2])

    venue_name = pub.get("venue", "")

    global crossref_mapping
    if venue_dblp not in crossref_mapping:
        crossref_mapping[venue_dblp] = []
    crossref_mapping[venue_dblp].append(pub)

    # Determine rank using the normalized dblp/acronym lookup
    rank, short_paper, acronym_row = classify_paper.core_rank(venue_name, venue_crossref, venue_dblp, year)

    # If the core table provides a fullpaper threshold for this venue, use it
    threshold = None
    if acronym_row is not None:
        try:
            if hasattr(acronym_row, 'get'):
                thr = acronym_row.get('fullpaper_threshold')
            else:
                thr = None
            if thr is not None and str(thr).strip() != "":
                threshold = int(str(thr).strip())
        except Exception:
            threshold = None

    if not short_paper and threshold is not None:
        pages_str = pub.get("pages", "")
        pagenum = classify_paper.get_paper_length(pages_str)
        if pagenum is None:
            print(f"No page number for {pub}")
        elif pagenum < threshold:
            short_paper = True

    return rank, short_paper, acronym_row

with open(os.path.join('..', 'proceedings.json'), 'r', encoding='utf-8') as f:
    data = json.load(f)

core_table = pd.read_csv(os.path.join("../inputs", "core_table.csv"))

rank_number = {}
found_venues = {}
found_crossrefs = {}
crossref_to_name = {}

for key, venues in data.items():
    for year, venue in venues.items():
        crossrefs=venue.get("key").split('/')
        crossref_short=f"{crossrefs[0]}/{crossrefs[1]}"
        pub = {
            "type": venue.get("type"),
            "authors": [],
            "title": venue.get("title"),
            "year": venue.get("year"),
            "url": "",
            "venue": venue.get("venue"),
            "crossref": crossref_short,
            "pages": "1-12"
        }
        if pub.get("venue") is None:
            #print(f"Missing venue for {pub}")
            continue

        found_crossrefs[crossref_short]=1
        desc = f"{pub.get("venue", "")} {pub.get("title", "")}"
        if crossref_short not in crossref_to_name:
            crossref_to_name[crossref_short] = desc
        else:
            same=False
            for publ in crossref_to_name.get(crossref_short, "").split('\n'):
                if publ == desc:
                    same=True
            if not same:
                crossref_to_name[crossref_short] +='\n' + desc
        rank, short_paper, acronym_row = get_core_rank(pub)
        #print(f"{pub['venue']} ({pub['year']}): {rank} - {short_paper}")
        rank_str = f"{rank} - {'short' if short_paper else 'full'} number of {venue.get('type')}"
        rank_number[rank_str] = rank_number.get(rank_str, 0) + 1
        if acronym_row is not None:
            id = acronym_row.get("ID")
            if acronym_row is not None and id is not None:
                id_str=str(id).upper()
                if id_str not in found_venues:
                    found_venues[id_str] = []
                found_venues[id_str].append(pub)



total = sum(rank_number.values()) if rank_number else 0
for rank_str, count in sorted(rank_number.items(), key=lambda x: x[0]):
    pct = (count / total * 100) if total else 0.0
    print(f"{rank_str}: {count} ({pct:.1f}%)")



# Check core_table for missing IDs and count by computed rank
no_dblps = 0
missing_counts = {}
found_counts = {}
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    dblp_venues=row.get("dblp_venue")
    # Treat NaN as empty: require non-NaN and non-blank string
    if pd.notna(dblp_venues) and str(dblp_venues).strip() != "":
        for dblp_venue in str(dblp_venues).split(';'):
            if dblp_venue not in found_crossrefs:
                found_crossrefs[dblp_venue]=0
            else:
                found_crossrefs[dblp_venue]=2
    else:
        no_dblps+=1
    best_rank= "no_rank"
    core_ranks_str = row.get("YearsListed", "")
    if "_A*" in core_ranks_str:
        best_rank = "A*"
    elif '_A' in core_ranks_str:
        best_rank = "A"
    elif '_B' in core_ranks_str:
        best_rank = "B"
    elif '_C' in core_ranks_str:
        best_rank = "C"
            
    cid = str(row.get("ID")).upper()
    if cid not in found_venues:
        if core_ranks_str!="ERA2010_A":
            missing_counts[best_rank] = missing_counts.get(best_rank, 0) + 1
            if best_rank == PRINT_RANK:
                if row['dblp_venue'] is not None and row['dblp_venue'] != "":
                    print(f"Missing {best_rank} venue: {row['dblp_venue']} {row.get('Acronym', '')} {row.get('Name', '')} {row.get('YearsListed')} (ID: {cid})")
                    venue_crossref = row.get("dblp_venue", "")
                    if venue_crossref in crossref_mapping:
                        print(f"Found in crossref_mapping: {venue_crossref} {row.get('Acronym', '')} {row.get('Name', '')}")
                        for pub in crossref_mapping[venue_crossref]:
                            print(f">{pub.get('venue')}< : {pub["title"]} ({pub['year']}) - {pub.get('type')})")
    else:
        found_counts[best_rank] = found_counts.get(best_rank, 0) + 1

print("\nMissing counts by rank category:")
for k, v in sorted(found_counts.items(), key=lambda x: x[0]):
    missing=missing_counts.get(k, 0)
    found=v
    total_k=missing+found
    print(f"{k}: {missing} ({missing/total_k*100:.1f}%) missing, {found} ({found/total_k*100:.1f}%) found, total {total_k}")


not_in_proceedings=[crossref for crossref, found in found_crossrefs.items() if found==0]
not_in_core_table=[crossref for crossref, found in found_crossrefs.items() if found==1 and crossref.startswith('conf/')]
print(f"Crossrefs not found in proceeding.json: {len(not_in_proceedings)}")
print(f"Crossrefs not found in core_table_csv: {len(not_in_core_table)}")
print(f"Core table rows with no dblp venue: {no_dblps}")

print(f"\nNot in proceedings.json: {not_in_proceedings}")
print(f"\nNot in core_table_csv: {not_in_core_table}")

A - full number of proceedings: 5616 (9.0%)
A - short number of proceedings: 3440 (5.5%)
A* - full number of proceedings: 2394 (3.8%)
A* - short number of proceedings: 2470 (4.0%)
B - full number of proceedings: 7941 (12.7%)
B - short number of proceedings: 2997 (4.8%)
C - full number of proceedings: 6051 (9.7%)
C - short number of proceedings: 1773 (2.8%)
TBR - full number of proceedings: 2 (0.0%)
no_rank - full number of proceedings: 29460 (47.2%)
no_rank - short number of proceedings: 330 (0.5%)

Missing core_table entries:

Missing counts by rank category:
A: 17 (5.2%) missing, 310 (94.8%) found, total 327
A*: 0 (0.0%) missing, 84 (100.0%) found, total 84
B: 178 (29.1%) missing, 433 (70.9%) found, total 611
C: 509 (49.3%) missing, 524 (50.7%) found, total 1033
no_rank: 18 (27.7%) missing, 47 (72.3%) found, total 65
Crossrefs not found in proceeding.json: 10
Crossrefs not found in core_table_csv: 5157
Core table rows with no dblp venue: 866

Not in proceedings.json: ['journals/tog',

In [3]:
import os
import json

counts_path = os.path.join('..', 'dblp_venue_counts.json')
try:
    with open(counts_path, 'r', encoding='utf-8') as f:
        dblp_counts = json.load(f)
except Exception as e:
    print(f"Nem sikerült betölteni {counts_path}: {e}")
    dblp_counts = {}

# A korábbi cellában létrehozott változó: not_in_core_table
not_in = globals().get('not_in_core_table')
if not_in is None:
    print("Figyelmeztetés: 'not_in_core_table' változó nem található a kernelben. Üres listát használok.")
    not_in = []

# Párosítás (crossref_short -> count) és rendezés
rows = [(v, int(dblp_counts.get(v, 0))) for v in not_in]
rows_sorted = sorted(rows, key=lambda x: x[1], reverse=True)

# Statisztikák
total_missing = len(rows)
with_count = sum(1 for _, c in rows if c > 0)
sum_counts = sum(c for _, c in rows)

print(f"Összes 'not_in_core_table' elem: {total_missing}")
print(f"Ebből DBLP-ben talált (count>0): {with_count}")
print(f"DBLP-ben összes előfordulás összesen: {sum_counts}")

# Top 50
top50 = rows_sorted[:50]
print('\nTop 50 leggyakoribb not_in_core_table elemek (crossref_short: count):')
for v, c in top50:
    print(f"{v}: {c} \n{crossref_to_name.get(v, 'N/A')}")

# Mentsük el az eredményt a results mappába
os.makedirs('results', exist_ok=True)
out_path = os.path.join('results', 'not_in_core_table_sorted_by_dblp_counts.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump({'stats': {'total': total_missing, 'with_count': with_count, 'sum_counts': sum_counts}, 'top50': top50, 'all': rows_sorted}, f, ensure_ascii=False, indent=2)

print(f"\nRendezett lista és statisztika elmentve: {out_path}")

Összes 'not_in_core_table' elem: 5157
Ebből DBLP-ben talált (count>0): 5145
DBLP-ben összes előfordulás összesen: 1298702

Top 50 leggyakoribb not_in_core_table elemek (crossref_short: count):
conf/amcc: 24284 
ACC American Control Conference, ACC 2010, Baltimore, Maryland, USA, June 30 - July 2, 2010
ACC Proceedings of the 2004 American Control Conference, ACC 2004, Boston, MA, USA, June 30 - July 2, 2004
ACC American Control Conference, ACC 2024, Toronto, ON, Canada, July 10-12, 2024
ACC American Control Conference, ACC 2022, Atlanta, GA, USA, June 8-10, 2022
ACC 2021 American Control Conference, ACC 2021, New Orleans, LA, USA, May 25-28, 2021
ACC American Control Conference, ACC 2013, Washington, DC, USA, June 17-19, 2013
ACC 2025 American Control Conference, ACC 2025, Denver, CO, USA, July 8-10, 2025
ACC American Control Conference, ACC 2002, Anchorage, Alaska, USA, May 8-10 2002.
ACC American Control Conference, ACC 2023, San Diego, CA, USA, May 31 - June 2, 2023
ACC American Cont

In [ ]:
import os
import json
import re
import pandas as pd
import difflib
import requests

# Load proceedings and core table
with open(os.path.join('..', 'proceedings.json'), 'r', encoding='utf-8') as f:
    data = json.load(f)
core_table = pd.read_csv(os.path.join('../inputs', 'core_table.csv'))

# Stopwords for token comparison
stopwords = set([
    'and','of','the','in','on','for','with','international','conference',
    'symposium','proceedings','journal','transactions','workshop','letters',
    'systems','system','applications','application','computing','computers',
    'review','annals','bulletin','ieee','acm','springer','elsevier','was',
    'joint','congress','society','education','association',
])

# Stricter thresholds
JACCARD_THRESH = 0.15
FRAC_TITLE_THRESH = 0.25
FUZZY_THRESH = 0.85
MAX_FOUND = 100
print(f"Using stricter thresholds: jaccard={JACCARD_THRESH}, frac_title={FRAC_TITLE_THRESH}, fuzzy={FUZZY_THRESH}")

# Use LLM if OPENAI_API_KEY is set
USE_LLM = bool(os.getenv('OPENAI_API_KEY'))
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')

# Embedding model settings
EMB_MODEL = os.getenv('EMB_MODEL', 'all-MiniLM-L6-v2')
EMB_AVAILABLE = False
_emb_model = None
_core_embs = None

found_venues = set()
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    if row['dblp_venue']:
        found_venues.add(row['dblp_venue'])

def tokenize(text):
    parts = re.split(r"[^\wÀ-ž]+", (text or ''))
    toks = set()
    for p in parts:
        p = p.strip().lower()
        if not p:
            continue
        if len(p) <= 2:
            continue
        if p in stopwords:
            continue
        toks.add(p)
    return toks


def fuzzy_similar_count(set_a, set_b, thresh=FUZZY_THRESH, max_tokens=8):
    """Return number of fuzzy token pairs with similarity >= thresh.

    Compare up to `max_tokens` tokens from each set to keep runtime bounded.
    """
    if not set_a or not set_b:
        return 0
    a_list = sorted(list(set_a))[:max_tokens]
    b_list = sorted(list(set_b))[:max_tokens]
    count = 0
    for a in a_list:
        for b in b_list:
            if difflib.SequenceMatcher(None, a, b).ratio() >= thresh:
                count += 1
                if count >= 5:
                    return count
    return count


def ask_llm(item, api_key=None, model=None, timeout=20):
    api_key = api_key or os.getenv('OPENAI_API_KEY')
    if not api_key:
        return {'error': 'no_api_key'}
    model = model or LLM_MODEL

    proceedings = {
        'key': item.get('proceedings_key'),
        'year': item.get('year'),
        'title': item.get('title'),
        'venue_value': item.get('venue_value'),
        'crossref_short': item.get('crossref_short'),
    }
    core = {
        'core_name': item.get('core_name'),
        'core_acronym': item.get('core_acronym'),
        'core_dblp_venue': item.get('core_dblp_venue'),
        'common_tokens': item.get('common_tokens'),
        'jaccard': item.get('jaccard'),
        'common_count': item.get('common_count')
    }

    prompt = (
        "Decide whether the following Proceedings entry and Core table entry refer to the same conference or series. "
        "Answer in JSON format exactly: {\"same\": \"yes|no|maybe\", \"score\": <0.0-1.0>, \"reason\": \"...\"}.\n\n"
        f"Proceedings: {json.dumps(proceedings, ensure_ascii=False)}\n\n"
        f"Core: {json.dumps(core, ensure_ascii=False)}\n\n"
        "Consider title tokens, acronyms, dblp crossref, and similarity metrics. Be concise."
    )

    url = 'https://api.openai.com/v1/chat/completions'
    headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant that classifies whether two venue records refer to the same conference.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 200,
        'temperature': 0.0,
    }

    try:
        r = requests.post(url, headers=headers, json=payload, timeout=timeout)
        r.raise_for_status()
        j = r.json()
        text = j['choices'][0]['message']['content']
        try:
            out = json.loads(text)
            return out
        except Exception:
            import re as _re
            m = _re.search(r"\{.*\}", text, _re.DOTALL)
            if m:
                try:
                    return json.loads(m.group(0))
                except Exception:
                    return {'error': 'parse_failed', 'raw': text}
            return {'error': 'no_json', 'raw': text}
    except Exception as e:
        return {'error': 'request_failed', 'detail': str(e)}


# LLM comparison cell (prints concise JSON samples)
def compare_with_llm(suspicious_list=None, max_show=20):
    lst = suspicious_list if suspicious_list is not None else globals().get('suspicious', [])
    cnt = len(lst)
    print(f'compare_with_llm: {cnt} items', flush=True)
    if cnt == 0:
        return
    to_show = min(max_show, cnt)
    for i, s in enumerate(lst[:to_show]):
        print('---', flush=True)
        print(f'[{i+1}/{cnt}] Proceedings: {s.get("proceedings_key", "")} / {s.get("year", "")}', flush=True)
        try:
            print(json.dumps(s, ensure_ascii=False, indent=2), flush=True)
        except Exception:
            print(repr(s), flush=True)

        if USE_LLM:
            res = ask_llm(s)
            try:
                print('LLM verdict:', json.dumps(res, ensure_ascii=False, indent=2), flush=True)
            except Exception:
                print('LLM verdict:', repr(res), flush=True)
        elif EMB_AVAILABLE:
            res = ask_llm_embedding(s)
            try:
                print('Embedding verdict:', json.dumps(res, ensure_ascii=False, indent=2), flush=True)
            except Exception:
                print('Embedding verdict:', repr(res), flush=True)
        else:
            print('LLM/Embedding not enabled (set OPENAI_API_KEY or install sentence-transformers)', flush=True)

    if cnt > to_show:
        print(f'... and {cnt - to_show} more', flush=True)


# Precompute token sets for core_table rows
core_rows = []
for i, row in core_table.iterrows():
    name = row.get('Name', '')
    skip=False
    for dblp_venue in str(row.get('dblp_venue', '')).split(';'):
        if dblp_venue.strip() in found_venues:
            skip=True
    if skip:
        continue 
    toks = tokenize(name)
    core_rows.append({
        'index': i,
        'Name': name,
        'Acronym': str(row.get('Acronym', '')).strip(),
        'dblp_venue': str(row.get('dblp_venue', '')).strip(),
        'tokens': toks,
        'row': row
    })

# Try to load embedding model and precompute core embeddings
try:
    from sentence_transformers import SentenceTransformer, util
    _emb_model = SentenceTransformer(EMB_MODEL)
    core_texts = [cr['Name'] or '' for cr in core_rows]
    if core_texts:
        _core_embs = _emb_model.encode(core_texts, convert_to_tensor=True)
    EMB_AVAILABLE = True
    print(f'Embedding model loaded ({EMB_MODEL}), precomputed {len(core_texts)} core embeddings', flush=True)
except Exception as e:
    EMB_AVAILABLE = False
    _emb_model = None
    _core_embs = None
    print('Embedding model not available:', e, flush=True)


def ask_llm_embedding(item, top_k=1):
    if not EMB_AVAILABLE or _emb_model is None or _core_embs is None:
        return {'error': 'embeddings_unavailable'}
    proc_text = ' '.join(filter(None, [str(item.get('title', '')), str(item.get('venue_value', '')), str(item.get('crossref_short', ''))]))
    proc_emb = _emb_model.encode(proc_text, convert_to_tensor=True)
    scores = util.cos_sim(proc_emb, _core_embs)[0]
    best_idx = int(scores.argmax())
    best_score = float(scores[best_idx])
    best_core = core_rows[best_idx]
    if best_score >= 0.72:
        same = 'yes'
    elif best_score >= 0.55:
        same = 'maybe'
    else:
        same = 'no'
    reason = f"embedding_score={best_score:.3f}; core_name='{best_core['Name']}'"
    return {'same': same, 'score': best_score, 'reason': reason, 'best_core_index': best_idx}


# For each core_table row that lacks a dblp_venue, search proceedings.json for candidate DBLP records
core_missing = []
for i, row in core_table.iterrows():
    dblp = row.get('dblp_venue')
    if pd.isna(dblp) or str(dblp).strip() == '':
        core_missing.append((i, row))

print(f'Searching candidates for {len(core_missing)} core rows missing dblp_venue')
matches_for_core = {}
# Flatten proceedings entries for iteration
entries = [(k, y, v) for k, vs in data.items() for y, v in vs.items()]
# Progress wrapper reuse (tqdm may or may not be available)
try:
    from tqdm import tqdm
    progress = lambda it, **kw: tqdm(it, **kw)
except Exception:
    try:
        from tqdm.notebook import tqdm
        progress = lambda it, **kw: tqdm(it, **kw)
    except Exception:
        progress = lambda it, **kw: it

for idx, row in progress(core_missing, desc='Core missing -> scanning', unit='core'):
    core_name = str(row.get('Name',''))
    core_acro = str(row.get('Acronym','')).strip().lower()
    core_toks = tokenize(core_name)
    candidates = []
    checked = set()
    for proc_key, year, v in entries:
        proc_title = v.get('title','') or ''
        proc_venue = v.get('venue','') or ''
        proc_cross = v.get('key','') or v.get('crossref','') or ''
        proc_short = '/'.join(str(proc_cross).split('/')[:2]) if proc_cross else ''
        # skip if crossref already associated to a known core venue
        if proc_short in found_venues:
            continue
        if proc_short in checked:
            continue
        checked.add(proc_short)
        text = ' '.join([proc_title, proc_venue])
        text_toks = tokenize(text)
        if not text_toks:
            continue
        common = core_toks & text_toks
        union = core_toks | text_toks
        common_count = len(common)
        jaccard = (len(common) / len(union)) if union else 0.0
        fuzzy_matches = 0
        if jaccard < JACCARD_THRESH and len(core_toks) * len(text_toks) <= 200:
            fuzzy_matches = fuzzy_similar_count(core_toks, text_toks)
        # combined score (weights chosen heuristically)
        score = 0.6 * jaccard + 0.25 * min(fuzzy_matches,5)/5 + 0.15 * min(common_count,5)/5
        # prefer matches where acronym equals crossref second tag or venue matches acronym
        bonus = 0.0
        if '/' in proc_short:
            parts = proc_short.split('/')
            if len(parts) > 1 and parts[1].lower() == core_acro:
                bonus += 0.15
        if core_acro and core_acro == str(proc_venue).strip().lower():
            bonus += 0.10
        final_score = min(1.0, score + bonus)
        if final_score >= 0.12:
            candidates.append({'proceedings_key': proc_key, 'year': year, 'title': proc_title, 'venue': proc_venue, 'crossref_short': proc_short, 'jaccard': round(jaccard,3), 'fuzzy': int(fuzzy_matches), 'common': common_count, 'score': round(final_score,3)})
    # sort and keep top 10
    candidates_sorted = sorted(candidates, key=lambda x: x['score'], reverse=True)[:10]
    matches_for_core[idx] = {'core_id': int(row.get('ID')) if 'ID' in row else idx, 'core_name': core_name, 'core_acronym': core_acro, 'candidates': candidates_sorted}
    # If we found candidates for this core row, print them immediately (top 3 for quick inspection)
    if candidates_sorted:
        try:
            core_id_display = int(row.get('ID')) if 'ID' in row and pd.notna(row.get('ID')) else idx
        except Exception:
            core_id_display = idx
        print(f"[MATCH] Core {core_id_display} - {core_acro} - {core_name} -> {len(candidates_sorted)} candidate(s). Top score={candidates_sorted[0].get('score')}")
        try:
            print(json.dumps(candidates_sorted, ensure_ascii=False, indent=2))
        except Exception:
            print(candidates_sorted)

# Print summary and save results
total_found = sum(1 for v in matches_for_core.values() if v['candidates'])
print(f'Found candidate matches for {total_found}/{len(core_missing)} core rows (showing up to 10 each)')
os.makedirs('results', exist_ok=True)
out_path = os.path.join('results', 'core_missing_candidate_matches.json')
with open(out_path, 'w', encoding='utf-8') as fh:
    json.dump({'meta': {'num_core_missing': len(core_missing), 'with_candidates': total_found}, 'matches': matches_for_core}, fh, ensure_ascii=False, indent=2)
print(f'Wrote candidate matches to {out_path}')


Using stricter thresholds: jaccard=0.15, frac_title=0.25, fuzzy=0.85

Missing core_table entries:


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6278.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded (all-MiniLM-L6-v2), precomputed 881 core embeddings
Searching candidates for 866 core rows missing dblp_venue


Core missing -> scanning:   0%|          | 1/866 [00:01<26:43,  1.85s/core]

[MATCH] Core 141 - apcomin - Application of Computers and Operations Research in the Minerals Industries -> 10 candidate(s). Top: conf/or score=0.196
[
  {
    "proceedings_key": "conf/or",
    "year": "2010",
    "title": "Operations Research Proceedings 2010, Selected Papers of the Annual International Conference of the German Operations Research Society, Universität der Bundeswehr, München, September 1-3, 2010",
    "venue": "OR",
    "crossref_short": "conf/or",
    "jaccard": 0.143,
    "fuzzy": 1,
    "common": 2,
    "score": 0.196
  },
  {
    "proceedings_key": "conf/scor",
    "year": "2016",
    "title": "5th Student Conference on Operational Research, SCOR 2016, Nottingham, UK, April 8-10, 2016",
    "venue": "SCOR",
    "crossref_short": "conf/scor",
    "jaccard": 0.091,
    "fuzzy": 2,
    "common": 1,
    "score": 0.185
  },
  {
    "proceedings_key": "conf/door",
    "year": "2016",
    "title": "Discrete Optimization and Operations Research - 9th International Confere

...

Core missing -> scanning:   7%|▋         | 62/866 [01:40<19:43,  1.47s/core]

[MATCH] Core 932 - nan - International Conference on Coastal Engineering -> 10 candidate(s). Top: conf/sede score=0.168
[
  {
    "proceedings_key": "conf/sede",
    "year": "2010",
    "title": "ISCA 19th International Conference on Software Engineeringand Data Engineering (SEDE-2010) June 16-18, 2010, Hilton Fisherman's Wharf, San Francisco, CA, USA",
    "venue": "SEDE",
    "crossref_short": "conf/sede",
    "jaccard": 0.062,
    "fuzzy": 2,
    "common": 1,
    "score": 0.168
  },
  {
    "proceedings_key": "conf/sysose",
    "year": "2010",
    "title": "5th International Conference on System of Systems Engineering, SoSE 2010, Loughborough, UK, 22-24 June, 2010",
    "venue": "SoSE",
    "crossref_short": "conf/sysose",
    "jaccard": 0.143,
    "fuzzy": 1,
    "common": 1,
    "score": 0.166
  },
  {
    "proceedings_key": "conf/jinis",
    "year": "2023",
    "title": "Proceedings of the International Conference on Systems Engineering, Arequipa, Peru, October 03-05, 2023.",
   

Core missing -> scanning:   7%|▋         | 63/866 [01:41<18:00,  1.35s/core]

[MATCH] Core 940 - nan - International Conference on Composite Materials -> 1 candidate(s). Top: conf/fomms score=0.123
[
  {
    "proceedings_key": "conf/fomms",
    "year": "2015",
    "title": "Molecular Modeling and the Materials Genome: Foundations of Molecular Modeling and Simulation 2015, FOMMS 2015, The Resort at the Mountains, Mt. Hood, OR, USA, July 12-16, 2015",
    "venue": "FOMMS",
    "crossref_short": "conf/fomms",
    "jaccard": 0.071,
    "fuzzy": 1,
    "common": 1,
    "score": 0.123
  }
]


Core missing -> scanning:   7%|▋         | 64/866 [01:42<16:53,  1.26s/core]

In [5]:
MAX_FOUND = 100
# Find suspicious matches with stricter heuristic
acronyms = {}
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    dblp = row.get('dblp_venue')
    # consider rows missing dblp_venue (NaN or blank)
    if pd.notna(dblp) and str(dblp).strip() != '':
        continue
    acr_col = row.get('Acronym')
    if pd.isna(acr_col) or str(acr_col).strip() == '':
        continue
    for acronym in str(acr_col).split(';'):
        a = acronym.strip().lower()
        if a:
            acronyms[a] = {'Name': row.get('Name'), 'Acronym': row.get('Acronym'), 'YearsListed': row.get('YearsListed')}

suspicious = []
stop = False
printed_crossref=[]
for key, venues in data.items():
    if stop:
        break
    for year, v in venues.items():
        title = v.get('title', '')
        venue_val = v.get('venue', '') or ''
        venue_norm = str(venue_val).strip().lower()
        crossref = v.get('key', '') or v.get('crossref', '')
        crossref_short = '/'.join(str(crossref).split('/')[:2]) if crossref else ''
        if crossref_short in found_venues:
            continue
        if crossref_short in printed_crossref:
            continue
        if venue_norm and venue_norm in acronyms:
            cr = acronyms[venue_norm]
            suspicious.append({
                'proceedings_key': key,
                'year': year,
                'title': title,
                'venue_value': venue_val,
                'crossref_short': crossref_short,
                'core_name': cr.get('Name'),
                'core_acronym': cr.get('Acronym'),
                'core_yearslisted': cr.get('YearsListed')
            })
            printed_crossref.append(crossref_short)
            if len(suspicious) >= MAX_FOUND:
                stop = True
                break

print(f'Found {len(suspicious)} suspicious pairs to review (limited to {MAX_FOUND})', flush=True)

try:
    compare_with_llm(suspicious, max_show=MAX_FOUND)
except Exception as e:
    print(f'compare_with_llm not available or failed: {e}', flush=True)



Missing core_table entries:
Found 25 suspicious pairs to review (limited to 100)
compare_with_llm: 25 items
---
[1/25] Proceedings: conf/gi-ws / 2023
{
  "proceedings_key": "conf/gi-ws",
  "year": "2023",
  "title": "IEEE/ACM International Workshop on Genetic Improvement, GI@ICSE 2023, Melbourne, Australia, May 20, 2023",
  "venue_value": "GI",
  "crossref_short": "conf/gi-ws",
  "core_name": "IEEE Global Internet Symposium",
  "core_acronym": "GI",
  "core_yearslisted": "ERA2010_C, CORE2013_C, CORE2014_C, CORE2017_C, CORE2018_C, CORE2020_C, CORE2021_C, CORE2023_C, ICORE2026_C"
}
Embedding verdict: {
  "same": "no",
  "score": 0.42467033863067627,
  "reason": "embedding_score=0.425; core_name='The Annual International Conference of the IEEE Engineering in Medicine and Biology Society'",
  "best_core_index": 823
}
---
[2/25] Proceedings: conf/ais2 / 2010
{
  "proceedings_key": "conf/ais2",
  "year": "2010",
  "title": "Autonomous and Intelligent Systems - First International Conference

In [6]:
# Listázza azokat az eseteket, ahol a core `Acronym` megegyezik a crossref második tagjával (hatékony változat)
import os
import json
from collections import defaultdict

# Build found_venues set from core_table dblp_venue to skip already-known venues
found_venues = set()
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    if row['dblp_venue']:
        found_venues.add(row['dblp_venue'])
        

# Build acronym -> core rows map (lowercased) from precomputed core_rows for O(1) lookup
acro_map = defaultdict(list)
for i, cr in core_table.iterrows():
    a = str(cr.get('Acronym','')).strip().lower()
    if a:
        acro_map[a].append({'index': i, 'Acronym': cr.get('Acronym'), 'Name': cr.get('Name'), 'YearsListed': cr.get('YearsListed'), 'dblp_venue': cr.get('dblp_venue')})

# Flatten proceedings entries to a list for stable iteration and progress reporting
entries = [(k, y, v) for k, vs in data.items() for y, v in vs.items()]

# Progress wrapper: prefer console tqdm, fallback to notebook tqdm, else identity
try:
    from tqdm import tqdm
    progress = lambda it, **kw: tqdm(it, **kw)
except Exception:
    try:
        from tqdm.notebook import tqdm
        progress = lambda it, **kw: tqdm(it, **kw)
    except Exception:
        progress = lambda it, **kw: it

matches = []
for key, year, v in progress(entries, desc="Checking crossref-second", unit="entry"):
    crossref = v.get('key') or ''
    if not crossref:
        continue
    parts = str(crossref).split('/')
    if len(parts) < 2:
        continue
    if parts[0] != 'conf':
        continue
    short = f"{parts[0]}/{parts[1]}"
    if short in found_venues:
        continue
    second = parts[1].strip().lower()
    if second in acro_map:
        for core_row in acro_map[second]:
            matches.append({
                'proceedings_key': key,
                'year': year,
                'title': v.get('title'),
                'venue_value': v.get('venue'),
                'crossref_short': short,
                'crossref_second': second,
                'core_index': core_row['index'],
                'core_acronym': core_row.get('Acronym'),
                'core_name': core_row.get('Name'),
                'core_rank': core_row.get('YearsListed')
            })

print(f'Found {len(matches)} acronym == crossref-second matches')
# write results to results/acro_crossref_matches.json
os.makedirs('results', exist_ok=True)
out_path = os.path.join('results', 'acro_crossref_matches.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(matches, f, ensure_ascii=False, indent=2)
print(f'Wrote {len(matches)} matches to {out_path}', flush=True)


Missing core_table entries:


Checking crossref-second: 100%|██████████| 63000/63000 [00:00<00:00, 894341.18entry/s]

Found 816 acronym == crossref-second matches
Wrote 816 matches to results/acro_crossref_matches.json


In [7]:
# Scan core_table for duplicate non-empty `Acronym` values and print matches; also report empty Acronym rows
import pandas as pd
import numpy as np
# Normalize acronyms (lowercase, stripped)
acro_norm = core_table['Acronym'].fillna('').astype(str).str.strip().str.lower()
core_table['_acro_norm'] = acro_norm
# Consider only non-empty acronyms for duplicate detection
non_empty_mask = core_table['_acro_norm'] != ''
counts = core_table.loc[non_empty_mask, '_acro_norm'].value_counts()
dups = counts[counts > 1]
if dups.empty:
    print('No duplicated non-empty Acronym values found')
else:
    print(f"Found {len(dups)} duplicated non-empty acronym(s): {', '.join(dups.index.tolist())}")
    for ac in dups.index:
        print('\n=== Acronym:', ac, '===')
        display_cols = ['ID','Acronym','Name','YearsListed','dblp_venue']
        cols = [c for c in display_cols if c in core_table.columns]
        print(core_table[core_table['_acro_norm'] == ac][cols].to_string(index=False))
# Report rows with empty acronym separately
empty_rows = core_table[core_table['_acro_norm'] == '']
if empty_rows.shape[0] > 0:
    print(f"\nRows with empty Acronym: {empty_rows.shape[0]} (showing up to 20 rows)")
    display_cols = ['ID','Acronym','Name','YearsListed','dblp_venue']
    cols = [c for c in display_cols if c in core_table.columns]
    print(empty_rows[cols].head(20).to_string(index=False))
# cleanup temporary column
core_table.drop(columns=['_acro_norm'], inplace=True, errors='ignore')

Found 51 duplicated non-empty acronym(s): isc, icsc, icis, iccc, icvs, cec, caia, trb, sea, ipc, embc, apcom, iccsa, icci, hai, xp, cis, cca, grc, acsac, ie, ismm, inap, iswc, iccp, dsn, icec, ecrypt, itc, icspc, ices, cacsd, ai, icics, cisis, sac, dac, dicta, gi, ism, ais, sc, icannga, osdi, acac, isarc, edm, icmr, sigchi-nz, wcc, ias

=== Acronym: isc ===
  ID Acronym                                                               Name                                                                                                            YearsListed         dblp_venue
 827     ISC                                    Information Security Conference CORE2008_C, ERA2010_C, CORE2013_C, CORE2014_C, CORE2017_B, CORE2018_B, CORE2020_B, CORE2021_B, CORE2023_C, ICORE2026_C                NaN
 820     ISC                                   Industrial Simulation Conference             ERA2010_C, CORE2013_C, CORE2014_C, CORE2017_C, CORE2018_C, CORE2020_C, CORE2021_C, CORE2023_C, ICORE2026_C      

In [8]:
# Update `core_table.csv` from `results/acro_crossref_matches.json` for unique acronyms (only fill empty `dblp_venue`)
import os
import json
import pandas as pd
# Paths
matches_path = os.path.join('results','acro_crossref_matches_.json')
core_in_path = os.path.join('..','inputs','core_table.csv')
core_out_path = os.path.join('..','inputs','core_table_.csv')
if not os.path.exists(matches_path):
    print(f'Matches file not found: {matches_path}')
else:
    with open(matches_path,'r', encoding='utf-8') as f:
        matches = json.load(f)
    # Build map: normalized acronym -> set of crossref_short values
    acro_to_cross = {}
    for m in matches:
        ac = str(m.get('core_acronym') or '').strip().lower()
        cross = str(m.get('crossref_short') or '').strip()
        if not ac:
            continue
        acro_to_cross.setdefault(ac, set()).add(cross)
    # Load core table
    if not os.path.exists(core_in_path):
        print(f'Core table not found: {core_in_path}')
    else:
        df = pd.read_csv(core_in_path)
        df['_acro_norm'] = df['Acronym'].fillna('').astype(str).str.strip().str.lower()
        # find duplicated non-empty acronyms
        dup_counts = df.loc[df['_acro_norm']!='', '_acro_norm'].value_counts()
        dups = set(dup_counts[dup_counts>1].index.tolist())
        changes = []
        for ac, cross_set in sorted(acro_to_cross.items()):
            if ac in dups:
                print(f"Skipping {ac!r}: acronym is duplicated in core_table")
                continue
            # filter out empty crossref values
            nonempty = [c for c in sorted(cross_set) if c]
            if not nonempty:
                print(f"Skipping {ac!r}: no crossref_short available in matches")
                continue
            uniq = sorted(set(nonempty))
            if len(uniq) != 1:
                print(f"Skipping {ac!r}: multiple crossref_short values found: {uniq}")
                continue
            value = uniq[0]
            rows = df.index[df['_acro_norm'] == ac].tolist()
            if not rows:
                print(f"No core_table row with Acronym {ac!r} found")
                continue
            if len(rows) > 1:
                print(f"Skipping {ac!r}: unexpected multiple core rows {rows}")
                continue
            idx = rows[0]
            # only fill when existing dblp_venue is empty or NaN
            old_val = df.at[idx, 'dblp_venue'] if 'dblp_venue' in df.columns else None
            is_empty = pd.isna(old_val) or str(old_val).strip() == ''
            if is_empty:
                df.at[idx, 'dblp_venue'] = value
                changes.append((ac, int(df.at[idx, 'ID']) if 'ID' in df.columns else idx, old_val, value))
                print(f"Filled Acronym {ac!r} at row {idx}: dblp_venue {old_val!r} -> {value!r}")
            else:
                if f"conf/{ac.lower()}"!=old_val.lower(): 
                    print(f"Skipping update for {ac!r}: existing non-empty dblp_venue kept ({old_val!r})")
        # Write updated core table copy
        df.drop(columns=['_acro_norm'], inplace=True, errors='ignore')
        df.to_csv(core_out_path, index=False)
        print(f"Wrote updated core table to {core_out_path} ({len(changes)} changes)")
        if changes:
            print('Sample changes:')
            for c in changes[:20]:
                print(c)

Matches file not found: results/acro_crossref_matches_.json


In [9]:
json_path = Path('../dblp_venue_names.json')
if not json_path.exists():
    raise FileNotFoundError(f"{json_path} nem található.")
with open(json_path, 'r', encoding='utf-8') as fh:
    names = json.load(fh)

for crossref, name in names.items():
    rec = {"article": {
          "@key": "journals/cgf/abc24",
          "author": [],
          "title": "This should be classied as Rank A",
          "pages": "63-74",
          "year": "2020",
          "volume": "37",
          "journal": "Comput. Graph. Forum",
          "number": "2",
          "crossref": "journals/cgf/vc1",
        } }
    # rec= {"inproceedings": {
    #       "author": [
    #       ],
    #       "title": "This should be classied as Rank A*",
    #       "pages": "449-456",
    #       "year": "2003",
    #       "crossref": "conf/visualization/2003",
    #       "booktitle": "IEEE Visualization"
    #     }}
    key, record, rank, foreign_paper, short_paper, search_log, original_rank = classify_paper(rec)

NameError: name 'Path' is not defined

In [ ]:
# Iterate over conference_stats/dblp_venue_names.json and classify a synthetic record for each dblp venue
import json
import os
import classify_paper

in_path = os.path.join('..','dblp_venue_names.json')
try:
    with open(in_path, 'r', encoding='utf-8') as f:
        venue_names = json.load(f)
except Exception as e:
    print(f"Hiba a {in_path} betöltésekor: {e}")
    venue_names = {}

def ascii_histogram(d, width=60, cap=12):
    def to_int(k):
        if str(k) == "null":
            return 0
        try:
            return int(k)
        except:
            return 0

    # aggregálás
    agg = {}
    for k, v in d.items():
        i = to_int(k)
        if i >= cap:
            agg[f"{cap}+"] = agg.get(f"{cap}+", 0) + v
        else:
            agg[i] = agg.get(i, 0) + v

    max_val = max(agg.values())

    for k in sorted(agg, key=lambda x: cap if x == f"{cap}+" else int(x)):
        v = agg[k]
        n = int(v / max_val * width)
        bar = "#" * n
        print(f"{str(k):>3} | {bar} ({v})")

results = {}
count = 0
for dblp_key, acr_map in venue_names.items():
    for acr,pubnum_map in acr_map:
        # rec = {"inproceedings": {
        #     "author": [],
        #     "title": f"Test classification for {dblp_key}",
        #     "pages": "1-10",
        #     "year": "2020",
        #     "crossref": f"{dblp_key}/2020",
        #     "booktitle": acronym
        # }}
        #key, record, rank, foreign_paper, short_paper, search_log, original_rank = classify_paper.classify_paper(rec)
        #print(f"{dblp_key} {acr} ->  {key} {rank} {'short_paper' if short_paper else ''} {original_rank}") # {search_log}
        last_rank, short_paper, acronym_row =classify_paper.core_rank(acr, f"{dblp_key}/2020", f"{dblp_key}/2020", 2026)
        if acronym_row is not None:
            if last_rank=='A*':
                if not short_paper:
                    print(f"{dblp_key} >{acr}< ->  {last_rank} {'short' if short_paper else 'full'} |{acronym_row['Acronym']}| {acronym_row['Name']}")
                    ascii_histogram(pubnum_map,20)
        else:
            print(f"{dblp_key} >{acr}< ->  No match")

    count += 1

print(f"Processed {count} venues.")


conf/acl >ACL< ->  A* full |ACL;ACL/IJCNLP| Association for Computational Linguistics
 -1 |  (1)
  0 | #### (938)
  1 |  (40)
  2 |  (101)
  3 |  (118)
  4 | # (288)
  5 | #### (819)
  6 | ## (515)
  7 | #### (901)
  8 | ## (413)
  9 | #### (974)
 10 | ##### (1050)
 11 | ## (478)
12+ | #################### (4034)
conf/acl >ACL/IJCNLP< ->  A* full |ACL;ACL/IJCNLP| Association for Computational Linguistics
  0 |  (2)
  3 | ###### (93)
  4 |  (4)
  5 | # (24)
  6 | ## (35)
  7 | ## (36)
  8 | ######### (136)
  9 | #### (72)
 10 | ######### (139)
 11 | ###### (90)
12+ | #################### (295)
conf/ismar >ISMAR< ->  A* full |ISMAR;ISAR;IWAR| IEEE/ACM International Symposium on Mixed and Augmented Reality
  0 | #### (108)
  1 | ################# (414)
  2 |  (1)
  3 | ##### (123)
  4 |  (6)
  5 | ## (63)
  6 | # (34)
  7 | ## (68)
  8 | ###### (146)
  9 | #################### (471)
 10 | # (30)
 11 | # (39)
12+ |  (15)
conf/ismar >IWAR< ->  A* full |ISMAR;ISAR;IWAR| IEEE/ACM Internationa